# Training Dataset EDA
Exploratory analysis of the OptoJump annotation dataset used for model training.

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#3a3d4d",
    "text.color":       "#e0e0e0",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#a0a0b0",
    "ytick.color":      "#a0a0b0",
    "grid.color":       "#2a2d3a",
    "grid.linewidth":   0.6,
    "font.family":      "monospace",
})

ANNOTATIONS_CSV = "./data/output/annotations/optojump/ml_training_dataset.csv"
RECORDING_FPS   = 120

GREEN  = "#00e676"
RED    = "#ff5252"
YELLOW = "#ffd740"
BLUE   = "#40c4ff"
PURPLE = "#ce93d8"

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
raw = pd.read_csv(ANNOTATIONS_CSV)

# Derive athlete name, study id and test id from video_path
raw["study"]   = raw["video_path"].str.extract(r"study_(\d+)").astype(int)
raw["athlete"] = raw["video_path"].apply(
    lambda p: "_".join(os.path.splitext(os.path.basename(p))[0].split("_")[:-1])
)
raw["test_id"] = raw["video_path"].apply(
    lambda p: int(os.path.splitext(os.path.basename(p))[0].split("_")[-1])
)

# Label contact bouts (consecutive contact frames per video as one step)
contact = raw[raw["label"] == "contact"].copy()
contact["bout"] = (
    contact.groupby("video_path")["frame_number"]
    .transform(lambda x: (x.diff() != 1).cumsum())
)

# ── Train / test split (mirrors train_test_split in dataset.py: seed=42, 10%) ─
_athletes = sorted(raw["athlete"].unique())
_n_test   = int(len(_athletes) * 0.10)
_rng      = np.random.default_rng(42)
TEST_ATHLETES = sorted(_rng.choice(_athletes, _n_test, replace=False).tolist())
_test_set = set(TEST_ATHLETES)

raw_train = raw[~raw["athlete"].isin(_test_set)].copy()
raw_test  = raw[ raw["athlete"].isin(_test_set)].copy()

contact_train = contact[~contact["video_path"].isin(raw_test["video_path"])].copy()
contact_test  = contact[ contact["video_path"].isin(raw_test["video_path"])].copy()

print(f"Total  : {len(raw):,} frames  |  {raw['video_path'].nunique()} videos  |  {raw['athlete'].nunique()} athletes")
print(f"Train  : {len(raw_train):,} frames  |  {raw_train['video_path'].nunique()} videos  |  {raw_train['athlete'].nunique()} athletes")
print(f"Test   : {len(raw_test):,} frames  |  {raw_test['video_path'].nunique()} videos  |  {raw_test['athlete'].nunique()} athletes  ({', '.join(TEST_ATHLETES)})")

## 1 · Dataset overview

In [ ]:
def _subset_stats(df, cont_df, label):
    n_f       = len(df)
    n_contact = (df["label"] == "contact").sum()
    n_flight  = (df["label"] == "flight").sum()
    n_vid     = df["video_path"].nunique()
    n_ath     = df["athlete"].nunique()
    n_steps   = cont_df.groupby("video_path")["bout"].nunique().sum()
    return {
        "Subset":                 label,
        "Total frames":           f"{n_f:,}",
        "Contact frames":         f"{n_contact:,}  ({n_contact/n_f*100:.1f}%)",
        "Flight frames":          f"{n_flight:,}  ({n_flight/n_f*100:.1f}%)",
        "Videos":                 str(n_vid),
        "Athletes":               str(n_ath),
        "Contact bouts (steps)":  str(n_steps),
    }

summary = pd.DataFrame([
    _subset_stats(raw_train, contact_train, "Train"),
    _subset_stats(raw_test,  contact_test,  "Test"),
    _subset_stats(raw,       contact,       "Total"),
]).set_index("Subset")
summary.style.set_caption("Dataset summary — train / test / total")

## 2 · Flight / contact split

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

n_frames_train = len(raw_train)

# ── pie: overall split ────────────────────────────────────────────────────
ax = axes[0]
counts = raw_train["label"].value_counts()
ax.pie(
    counts,
    labels=[f"{l}\n{v:,} ({v/n_frames_train*100:.1f}%)" for l, v in counts.items()],
    colors=[GREEN, BLUE],
    startangle=90,
    wedgeprops=dict(linewidth=1.5, edgecolor="#0f1117"),
    textprops=dict(color="#e0e0e0", fontsize=11),
)
ax.set_title("Overall label split (train)", fontsize=13)

# ── bar: per-video contact % ──────────────────────────────────────────────
ax = axes[1]
video_contact_pct = (
    raw_train.groupby("video_path")["label"]
    .apply(lambda x: (x == "contact").mean() * 100)
    .sort_values()
)
ax.barh(range(len(video_contact_pct)), video_contact_pct.values,
        color=GREEN, alpha=0.8, height=0.8)
ax.axvline(video_contact_pct.mean(), color=YELLOW, lw=1.5, ls="--",
           label=f"mean {video_contact_pct.mean():.1f}%")
ax.set_xlabel("Contact frames (%)")
ax.set_title("Contact % per video (train)", fontsize=13)
ax.set_yticks([])
ax.legend()
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 3 · Athletes

In [ ]:
videos_per_athlete = (
    raw_train.groupby("athlete")["video_path"].nunique().sort_values(ascending=True)
)
n_athletes_train = raw_train["athlete"].nunique()

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(videos_per_athlete.index, videos_per_athlete.values,
               color=BLUE, alpha=0.85, height=0.7)
ax.bar_label(bars, padding=3, color="#e0e0e0", fontsize=9)
ax.set_xlabel("Number of videos")
ax.set_title(f"Videos per athlete — train set  (N = {n_athletes_train} athletes)", fontsize=13)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 4 · Steps per video

In [ ]:
steps_per_video = contact_train.groupby("video_path")["bout"].nunique().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── histogram ─────────────────────────────────────────────────────────────
ax = axes[0]
bins = range(1, steps_per_video.max() + 2)
counts, edges, patches = ax.hist(steps_per_video, bins=bins, color=PURPLE,
                                  alpha=0.85, edgecolor="#0f1117", align="left")
ax.axvline(steps_per_video.mean(), color=YELLOW, lw=1.5, ls="--",
           label=f"mean {steps_per_video.mean():.1f}")
ax.set_xlabel("Steps per video")
ax.set_ylabel("Number of videos")
ax.set_title("Distribution of steps per video (train)", fontsize=13)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

# ── per-athlete mean steps ────────────────────────────────────────────────
ax = axes[1]
video_athlete = raw_train[["video_path", "athlete"]].drop_duplicates()
steps_athlete = (
    steps_per_video.reset_index()
    .merge(video_athlete, on="video_path")
    .groupby("athlete")["bout"].mean()
    .sort_values(ascending=True)
)
bars = ax.barh(steps_athlete.index, steps_athlete.values,
               color=PURPLE, alpha=0.85, height=0.7)
ax.bar_label(bars, fmt="%.1f", padding=3, color="#e0e0e0", fontsize=9)
ax.set_xlabel("Mean steps per video")
ax.set_title("Mean steps per video by athlete (train)", fontsize=13)
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Steps per video — min: {steps_per_video.min()}, "
      f"median: {steps_per_video.median():.0f}, "
      f"mean: {steps_per_video.mean():.1f}, "
      f"max: {steps_per_video.max()}")

## 5 · Contact duration per step

In [ ]:
bout_lengths = (
    contact_train.groupby(["video_path", "bout"])["frame_number"]
    .count()
    .rename("frames")
)
bout_ms = bout_lengths / RECORDING_FPS * 1000

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(bout_ms, bins=40, color=GREEN, alpha=0.85, edgecolor="#0f1117")
ax.axvline(bout_ms.mean(), color=YELLOW, lw=1.5, ls="--",
           label=f"mean {bout_ms.mean():.0f} ms")
ax.axvline(bout_ms.median(), color=RED, lw=1.5, ls=":",
           label=f"median {bout_ms.median():.0f} ms")
ax.set_xlabel("Contact duration (ms)")
ax.set_ylabel("Number of steps")
ax.set_title("Distribution of contact duration per step (train)", fontsize=13)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Contact duration — min: {bout_ms.min():.0f} ms, "
      f"median: {bout_ms.median():.0f} ms, "
      f"mean: {bout_ms.mean():.0f} ms, "
      f"max: {bout_ms.max():.0f} ms")

## 6 · Left / right side balance

In [ ]:
# One row per contact bout with its side (train only)
bout_side = (
    contact_train.groupby(["video_path", "bout"])["side"]
    .first()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── overall side split (pie) ───────────────────────────────────────────────
ax = axes[0]
side_counts = bout_side["side"].value_counts()
ax.pie(
    side_counts,
    labels=[f"{s}\n{v} ({v/side_counts.sum()*100:.1f}%)" for s, v in side_counts.items()],
    colors=[BLUE, RED],
    startangle=90,
    wedgeprops=dict(linewidth=1.5, edgecolor="#0f1117"),
    textprops=dict(color="#e0e0e0", fontsize=11),
)
ax.set_title("Contact bouts by side (train)", fontsize=13)

# ── per-athlete left/right balance ────────────────────────────────────────
ax = axes[1]
video_athlete = raw_train[["video_path", "athlete"]].drop_duplicates()
athlete_side = (
    bout_side.merge(video_athlete, on="video_path")
    .groupby(["athlete", "side"])
    .size()
    .unstack(fill_value=0)
)
athlete_side = athlete_side.reindex(columns=["left", "right"], fill_value=0)
athlete_side = athlete_side.sort_values("left")
athlete_side.plot(
    kind="barh", ax=ax, color=[BLUE, RED], alpha=0.85,
    width=0.7, edgecolor="#0f1117",
)
ax.set_xlabel("Number of contact bouts")
ax.set_title("Left vs right bouts per athlete (train)", fontsize=13)
ax.legend(title="Side")
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# Feature Selection — XGBoost ablation (train set only)

Results are loaded from `stage1_baselines.json`, which was computed using
leave-one-athlete-out cross-validation on the **train split only**.

In [ ]:
with open('./data/output/gait/stage1/results.json', 'r') as f:
    data = json.load(f)

In [ ]:
models = ['kinematic', *[f"ablation_{key}" for key in data['xgboost_ablation'].keys()]]

accuracies = [
    data['kinematic']['accuracy'],
    *[
        data['xgboost_ablation'][key]['accuracy'] for key in data['xgboost_ablation'].keys()
    ]
]

plt.figure(figsize=(12, 6))
sns.barplot(x=models, y=accuracies)
plt.title('Overall Model Accuracy Comparison', fontsize=15)
plt.ylabel('Accuracy Score')
plt.xticks(rotation=45)
plt.ylim(max(0, min(accuracies) - 0.1), 1.0)
plt.show()

In [ ]:
f1_scores = [
    data['kinematic']['f1']['macro'],
    *[
        data['xgboost_ablation'][key]['f1']['macro'] for key in data['xgboost_ablation'].keys()
    ]
]

plt.figure(figsize=(12, 6))
sns.barplot(x=models, y=f1_scores)
plt.title('Overall Model Accuracy Comparison', fontsize=15)
plt.ylabel('F1-macro Score')
plt.xticks(rotation=45)
plt.ylim(max(0, min(f1_scores) - 0.1), 1.0)
plt.show()

In [ ]:
def plot_cm(cm, title):
    labels = ['Left Stance', 'Right Stance', 'Flight']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, cbar=False)
    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

model = models[-1]
if 'ablation' in model:
    model_data = data['xgboost_ablation'][model.replace('ablation_', '')]
else:
    model_data = data[model]
plot_cm(model_data['confusion_matrix'], ' '.join([i.capitalize() for i in model.split('_')]))

In [ ]:
importances = data['xgboost_ablation']['D_all']['feature_importances']
indices = np.arange(len(importances))

plt.figure(figsize=(12, 6))
plt.bar(indices, importances)
plt.title('XGBoost Feature Importance Ranking')
plt.xlabel('Feature Index')
plt.ylabel('Importance Score')

max_idx = np.argmax(importances)
plt.show()

### Features:
- 0–5   : norm. y-positions  (L/R heel, big_toe, ankle)
- 6–11  : y-velocities
- 12–15 : x-velocities
- 16–19 : joint angles
- 20–21 : hip y + hip dy/dt

In [ ]:
timing = data['kinematic']['timing_error']
labels = list(timing.keys())
ms_values = [v['ms'] for v in timing.values()]

plt.figure(figsize=(10, 5))
sns.barplot(x=labels, y=ms_values)
plt.title('Timing Error (ms) by Gait Event - Kinematic Model')
plt.ylabel('Milliseconds (ms)')
plt.show()